In [ ]:
# Cell 1: 

!pip -q install -U unsloth 
!pip install -q sacrebleu rouge-score entmax
!pip install -q unbabel-comet --no-deps
print("✅ Packages installed. Please RESTART the runtime now (Runtime → Restart session).")

In [ ]:
# Cell 2: 

import os
import re
import time
import torch
import pandas as pd
import numpy as np
import sacrebleu
from unsloth import FastLanguageModel
from kaggle_secrets import UserSecretsClient
from datasets import load_dataset
from tqdm import tqdm
from rouge_score import rouge_scorer
from comet import download_model, load_from_checkpoint
user_secrets = UserSecretsClient()
os.environ["HF_TOKEN"] = user_secrets.get_secret("kaggle-glm-eval")

print("✅ Environment ready!")

In [ ]:
# Cell 3: 

# --- 1. Load Model ---
MODEL_NAME = "Qwen/Qwen3-8B"    # "Qwen/Qwen3-8B" "Qwen/Qwen3-4B" "Qwen/Qwen3-4B-Instruct-2507"
# MODEL_NAME = "zai-org/GLM-4-9B-0414" # "unsloth/GLM-4-9B-0414-bnb-4bit"    

SOURCE_LANG = "Mandarin"
TARGET_LANG = "English"   # "Cantonese"  "English" "Mandarin"

compute_dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print("compute_dtype:", compute_dtype)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

print("model loaded.")

# --- 2. LOAD TEST DATA ---

test_dataset = load_dataset("csv", data_files={"test": "/kaggle/input/datasets/flores-200/flores_200_en_cmn.csv"})["test"]
# test_dataset = load_dataset("csv", data_files={"test": "/kaggle/input/datasets/flores-200/flores_200_en_yue.csv"})["test"]


src_list = test_dataset["tgt"]
ref_list = test_dataset["src"]
hyp_list = []

In [ ]:
# Cell 4: 

# --- 3. PROMPT TEMPLATE ---

def build_translation_prompt(src, src_lang=SOURCE_LANG, tgt_lang=TARGET_LANG):
    return (
        "<|im_start|>system\n"
        "You are a professional machine translation system.\n"
        f"Translate from {src_lang} to {tgt_lang}.\n"
        "Rules:\n"
        "1. Output ONLY the translation.\n"
        "2. Do NOT explain.\n"
        "3. Do NOT think aloud.\n"
        "4. Do NOT add notes or comments.\n"
        "5. Preserve the original meaning, tone, and style.\n"
        "6. Use natural, fluent target-language text.\n"
        "<|im_end|>\n"
        "<|im_start|>user\n"
        f"{src}\n"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        "/no_think\n"
    )


# --- 5. GENERATION LOOP ---
start_time = time.time()

for i, src in enumerate(tqdm(src_list)):
    prompt = build_translation_prompt(src, src_lang=SOURCE_LANG, tgt_lang=TARGET_LANG)
    
    inputs = tokenizer(prompt, return_tensors = "pt").to(model.device)

    outputs = model.generate(
        **inputs,
        max_new_tokens=256,  # 1024 for Qwen and 256 for GLM
        do_sample=False,
        renormalize_logits=True,
        use_cache=True,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )

    generated_tokens = outputs[:, inputs.input_ids.shape[1]:]
    translation = tokenizer.decode(
        generated_tokens[0],
        skip_special_tokens=True
    )
    translation = re.sub(
        r"<think>.*?</think>",
        "",
        translation,
        flags=re.DOTALL
    )
    translation = re.sub(r"</?think>", "", translation)
    cleaned_translation = "\n".join(
        dict.fromkeys(
            line for line in (l.strip() for l in translation.split('\n')) if line
        )
    )

    hyp_list.append(cleaned_translation.strip()) 

    if (i + 1) % 5 == 0:
        elapsed = time.time() - start_time
        per_sentence = elapsed / (i + 1)
        remaining = per_sentence * (len(src_list) - i - 1)
        print(f"[{i+1}/{len(src_list)}] "
              f"{per_sentence:.1f}s/sentence | "
              f"ETA: {remaining/60:.1f} min")

# --- 6. SAVE BASELINE RESULTS ---
pd.DataFrame({
    "src": src_list,
    "ref": ref_list,
    "hyp": hyp_list
}).to_csv("Qwen3-8B-cmn-en-results.csv", index=False)
print("✅ Base results saved to Qwen3-8B-cmn-en-results.csv")

In [ ]:
# Cell 5: Full Evaluation Script

# ====================== COMET ======================
print("Loading COMET model...")
model_path = download_model("Unbabel/wmt22-comet-da")

# Optional forward patch (you already saw it worked)
import comet.encoders.xlmr as xlmr_module
if hasattr(xlmr_module, "XLMREncoder"):
    def patched_forward(self, input_ids, attention_mask, **kwargs):
        outputs = self.model(input_ids=input_ids, attention_mask=attention_mask,
                             output_hidden_states=True, return_dict=True)
        return {
            "last_hidden_state": outputs.last_hidden_state,
            "all_layers": outputs.hidden_states
        }
    xlmr_module.XLMREncoder.forward = patched_forward
    print("✅ Forward patch applied")

comet_model = load_from_checkpoint(model_path)

comet_data = [{"src": s, "mt": h, "ref": r}
              for s, h, r in zip(src_list, hyp_list, ref_list)]

print("Running COMET prediction...")
comet_output = comet_model.predict(
    comet_data,
    batch_size=8,
    gpus=1 if torch.cuda.is_available() else 0,
    progress_bar=True
)

avg_comet = comet_output.system_score
print(f"✨ COMET Score: {avg_comet:.4f}")


# ====================== METRICS ======================
print("\nCalculating BLEU, ChrF...")

bleu = sacrebleu.corpus_bleu(hyp_list, [ref_list], tokenize='zh')
chrf = sacrebleu.corpus_chrf(hyp_list, [ref_list])

# ROUGE
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = [scorer.score(r, h) for r, h in zip(ref_list, hyp_list)]

avg_r1 = np.mean([s['rouge1'].fmeasure for s in rouge_scores])
avg_r2 = np.mean([s['rouge2'].fmeasure for s in rouge_scores])
avg_rl = np.mean([s['rougeL'].fmeasure for s in rouge_scores])

print(f"BLEU:  {bleu.score:.2f}")
print(f"ChrF:  {chrf.score:.2f}")
print(f"ROUGE-1: {avg_r1:.4f} | ROUGE-2: {avg_r2:.4f} | ROUGE-L: {avg_rl:.4f}")

# ====================== FINAL REPORT ======================
print("\n" + "="*50)
print("     Qwen3-8B cmn-en Final Evaluation")
print("="*50)
print(f"COMET Score:    {avg_comet:.4f}")
print(f"BLEU Score:     {bleu.score:.2f}")
print(f"ChrF Score:     {chrf.score:.2f}")
print(f"ROUGE-1:        {avg_r1:.4f}")
print(f"ROUGE-2:        {avg_r2:.4f}")
print(f"ROUGE-L:        {avg_rl:.4f}")
print("="*50)

# ====================== SAVE METRICS ======================
metrics_df = pd.DataFrame({
    "Metric": ["COMET", "BLEU", "ChrF", "ROUGE-1", "ROUGE-2", "ROUGE-L"],
    "Value": [avg_comet, bleu.score, chrf.score, avg_r1, avg_r2, avg_rl]
})
metrics_df.to_csv("Qwen3-8B-cmn-en-metrics.csv", index=False)
print("✅ Metrics saved to Qwen3-8B-cmn-en-metrics.csv")